# DeBERTa-NLI + mmBERT-base Fine-tune on MOOCCubeX CS

Notebook này làm đúng 2 stage bạn yêu cầu, không có GPT-5.4 trong loop:

1. **DeBERTa-v3-large-MNLI**: chấm direction `A -> B` vs `B -> A` cho edge bundle của mình.
2. **mmBERT-base fine-tune**: fine-tune binary prerequisite classifier trên **MOOCCubeX Computer Science prerequisites** rồi chấm strength cho edge bundle của mình.

Input bundle của project được tải từ **Google Drive link cũ**. Training data dùng **MOOCCubeX `prerequisites/cs.json`**. Notebook cũng cố tải thêm `entities/concept.json` để map concept id -> name/text; nếu không tải được thì vẫn cố parse trực tiếp từ `cs.json`.


In [27]:
# Input bundle from the same Drive link as before
DRIVE_URL = "https://drive.google.com/file/d/1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm/view?usp=sharing"
INPUT_PATH = "/content/edge_scoring_input_bundle_for_combo.json"

# Official-ish MOOCCubeX assets used for fine-tuning (CS only)
MOOCCUBEX_CS_URL = "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/prerequisites/cs.json"
MOOCCUBEX_CONCEPT_URL = "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/concept.json"
CS_JSON_PATH = "/content/mooccubex_cs.json"
CONCEPT_JSON_PATH = "/content/mooccubex_concept.json"

# Models
DEBERTA_MODEL_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
MMBERT_MODEL_ID = "jhu-clsp/mmBERT-base"

# Runtime knobs
MAX_LENGTH_NLI = 512
MAX_LENGTH_FT = 256
NLI_BATCH_SIZE = 8
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LR = 5e-5
WEIGHT_DECAY = 0.01
VAL_RATIO = 0.10
TEST_RATIO = 0.10
USE_LORA = True
SEED = 42

# Precision / stability
PREFER_BF16 = True
SKIP_NONFINITE_EVAL_BATCHES = True


# Imbalance handling
TRAIN_NEGATIVE_RATIO = 8
EVAL_NEGATIVE_RATIO = 100
MAX_TRAIN_ROWS = 10000
MAX_EVAL_ROWS = 12000
USE_CLASS_WEIGHTS = True
TUNE_THRESHOLD_ON_VAL = True

# Decision thresholds for the combined output
NLI_MARGIN_THRESHOLD = 0.15
FT_STRONG_THRESHOLD = 0.60
FT_WEAK_THRESHOLD = 0.40
COMBO_MARGIN_THRESHOLD = 0.12

# Output
OUTPUT_PATH = "/content/deberta_mmbert_mooccubex_cs_scores.json"
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/deberta_mmbert_mooccubex_cs_scores.json"

SHOW_PROGRESS = True


In [13]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors sentencepiece peft scikit-learn "requests==2.32.4" "protobuf>=5.29.1,<6"


In [14]:
import gc
import json
import math
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import requests
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, average_precision_score, precision_recall_curve, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")


def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown

    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return

    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)


def download_http_file(url: str, output_path: Path, timeout: int = 120) -> None:
    response = requests.get(url, timeout=timeout, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)


def load_json_flexible(path: Path) -> Any:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Empty file: {path}")

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            rows = []
            break
    if rows:
        return rows

    decoder = json.JSONDecoder()
    idx = 0
    objs = []
    length = len(text)
    while idx < length:
        while idx < length and text[idx].isspace():
            idx += 1
        if idx >= length:
            break
        obj, next_idx = decoder.raw_decode(text, idx)
        objs.append(obj)
        idx = next_idx
    if objs:
        return objs

    raise ValueError(f"Could not parse JSON/JSONL payload from {path}")


def load_json(path: Path) -> Any:
    return load_json_flexible(path)


def load_edge_bundle() -> dict[str, Any]:
    path = Path(INPUT_PATH)
    if path.exists() and path.stat().st_size > 0:
        print("Loading input bundle:", path)
        return load_json(path)

    file_id = extract_drive_file_id(DRIVE_URL)
    temp_download = Path("/content/edge_scoring_drive_download")
    if temp_download.exists():
        temp_download.unlink()
    download_from_drive(file_id, temp_download)
    print("Downloaded:", temp_download, "bytes=", temp_download.stat().st_size)

    if zipfile.is_zipfile(temp_download):
        extract_dir = Path("/content/edge_scoring_extract")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(temp_download, "r") as zf:
            zf.extractall(extract_dir)
        candidates = list(extract_dir.rglob("*.json"))
        if not candidates:
            raise FileNotFoundError("Downloaded zip has no JSON file")
        chosen = None
        for candidate in candidates:
            if "edge_scoring_input_bundle_for_combo" in candidate.name or "edge_scoring_input_bundle" in candidate.name:
                chosen = candidate
                break
        chosen = chosen or candidates[0]
        shutil.copyfile(chosen, path)
        print("Extracted JSON:", chosen)
    else:
        shutil.copyfile(temp_download, path)

    return load_json(path)


def ensure_mooccubex_files() -> tuple[Path, Path | None]:
    cs_path = Path(CS_JSON_PATH)
    concept_path = Path(CONCEPT_JSON_PATH)
    if not cs_path.exists() or cs_path.stat().st_size == 0:
        print("Downloading MOOCCubeX CS prerequisites from:", MOOCCUBEX_CS_URL)
        download_http_file(MOOCCUBEX_CS_URL, cs_path)
    if not concept_path.exists() or concept_path.stat().st_size == 0:
        try:
            print("Downloading MOOCCubeX concept catalog from:", MOOCCUBEX_CONCEPT_URL)
            download_http_file(MOOCCUBEX_CONCEPT_URL, concept_path)
        except Exception as exc:
            print("Concept catalog download failed; will parse without it:", repr(exc))
            concept_path = None
    return cs_path, concept_path


In [15]:
payload = load_edge_bundle()
clean_edges = payload.get("clean_candidate_edges") or []
pruned_edges = payload.get("pruned_edges") or []
global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
if not isinstance(clean_edges, list) or not isinstance(pruned_edges, list):
    raise ValueError("Expected clean_candidate_edges[] and pruned_edges[] in the edge bundle")

kp_index = {
    (kp.get("global_kp_id") or kp.get("kp_id")): kp
    for kp in global_kps
    if isinstance(kp, dict) and (kp.get("global_kp_id") or kp.get("kp_id"))
}


def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()


def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    return f"{name}. {desc}".strip()


def pair_text(edge: dict[str, Any], reverse: bool = False) -> tuple[str, str]:
    source_id = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target_id = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    return kp_text(source_id), kp_text(target_id)


print("Clean edges:", len(clean_edges))
print("Pruned edges:", len(pruned_edges))
print("KP catalog entries:", len(kp_index))
print("Sample edge:", clean_edges[0] if clean_edges else None)


Loading input bundle: /content/edge_scoring_input_bundle_for_combo.json
Clean edges: 93
Pruned edges: 20
KP catalog entries: 0
Sample edge: {'source_kp_id': 'kp_3d_datasets_and_ai_geometry_task_space', 'target_kp_id': 'kp_multi_view_and_voxel_based_3d_deep_learning', 'edge_scope': 'intra_course', 'provenance': 'llm_cross_check', 'keep_confidence': 'medium', 'keep_rationale': '3D datasets and AI+geometry task space provides prerequisite conceptual machinery, motivation, or implementation context needed to understand Multi-view and voxel-based 3D deep learning; the target specializes, applies, or operationalizes the source rather than merely co-occurring with it.', 'expected_directionality': 'moderate', 'review_status': 'optional', 'ready_for_modernbert': True}


## Load and normalize MOOCCubeX CS prerequisite data

Mục tiêu ở đây là dựng supervised pairs cho `mmBERT-base` từ **MOOCCubeX Computer Science prerequisites**. Vì schema thực tế của `cs.json` có thể thay đổi, parser dưới đây cố xử lý theo hướng chịu lỗi:
- nếu `cs.json` đã có text trực tiếp thì dùng luôn,
- nếu `cs.json` chủ yếu chứa concept IDs thì cố map qua `entities/concept.json`,
- nếu tập labels chỉ có positive edges thì tự tạo hard negatives bằng **reverse edges + corrupted target sampling**.


In [16]:
cs_path, concept_path = ensure_mooccubex_files()
raw_cs = load_json_flexible(cs_path)
print("CS prerequisite file:", cs_path, "type=", type(raw_cs).__name__)
if concept_path is not None and concept_path.exists():
    print("Concept catalog:", concept_path, "MB=", round(concept_path.stat().st_size / (1024 * 1024), 2))
else:
    print("Concept catalog: unavailable")


CS prerequisite file: /content/mooccubex_cs.json type= list
Concept catalog: /content/mooccubex_concept.json MB= 155.41


In [17]:
def load_concept_rows(path: Path | None) -> list[dict[str, Any]]:
    if path is None or not path.exists():
        return []
    parsed = load_json_flexible(path)
    if isinstance(parsed, list):
        return [row for row in parsed if isinstance(row, dict)]
    if isinstance(parsed, dict):
        rows = []
        for value in parsed.values():
            if isinstance(value, list):
                rows.extend(row for row in value if isinstance(row, dict))
            elif isinstance(value, dict):
                rows.append(value)
        return rows
    return []


def first_nonempty(row: dict[str, Any], keys: list[str]) -> Any:
    for key in keys:
        if key in row and row[key] not in (None, "", [], {}):
            return row[key]
    return None


def normalize_label(value: Any) -> int | None:
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, int):
        return 1 if value > 0 else 0
    if isinstance(value, float):
        return 1 if value > 0 else 0
    if isinstance(value, str):
        lowered = value.strip().lower()
        if lowered in {"1", "true", "yes", "y", "positive", "entailment", "prerequisite", "is_prerequisite"}:
            return 1
        if lowered in {"0", "false", "no", "n", "negative", "contradiction", "not_prerequisite", "not prerequisite"}:
            return 0
    return None


def coerce_text(value: Any) -> str | None:
    if isinstance(value, str):
        value = value.strip()
        return value or None
    if isinstance(value, list):
        items = [coerce_text(item) for item in value]
        items = [item for item in items if item]
        return "; ".join(items) if items else None
    return None


concept_rows = load_concept_rows(concept_path)
concept_index: dict[str, dict[str, Any]] = {}
for row in concept_rows:
    cid = first_nonempty(row, ["id", "concept_id", "ccid", "cid", "_id"])
    if cid is None:
        continue
    cid = str(cid)
    concept_index[cid] = {
        "name": coerce_text(first_nonempty(row, ["name", "title", "concept", "concept_name", "entity"])) or cid,
        "description": coerce_text(first_nonempty(row, ["description", "desc", "content", "definition", "summary"])),
        "raw": row,
    }

print("Loaded concept rows:", len(concept_rows))
print("Concept index size:", len(concept_index))


Loaded concept rows: 637572
Concept index size: 637572


In [18]:
SOURCE_ID_KEYS = [
    "source_id", "source", "source_concept_id", "from", "head", "src", "pre_id", "concept1_id", "c1_id", "left_id", "prerequisite_id"
]
TARGET_ID_KEYS = [
    "target_id", "target", "target_concept_id", "to", "tail", "dst", "post_id", "concept2_id", "c2_id", "right_id", "dependent_id"
]
SOURCE_TEXT_KEYS = [
    "source_name", "source_text", "source_concept", "from_name", "head_name", "pre_name", "concept1", "c1", "left_name", "prerequisite"
]
TARGET_TEXT_KEYS = [
    "target_name", "target_text", "target_concept", "to_name", "tail_name", "post_name", "concept2", "c2", "right_name", "dependent"
]
LABEL_KEYS = [
    "label", "gold_label", "human_label", "is_prerequisite", "prereq", "prerequisite_label", "relation_label", "y", "ground_truth"
]
NODE_ID_KEYS = ["id", "concept_id", "ccid", "cid", "_id", "conceptId"]
NODE_TEXT_KEYS = ["name", "title", "concept", "concept_name", "entity", "text", "label", "display_name"]
PREREQ_LIST_KEYS = [
    "prerequisites", "prerequisite", "prerequisite_ids", "pre_concepts", "preconcepts", "parents", "dependencies", "requires", "require", "pres"
]
CHILD_LIST_KEYS = [
    "successors", "children", "dependents", "next", "targets", "postrequisites", "outputs"
]


def concept_text_from_id(cid: str | None) -> str | None:
    if cid is None:
        return None
    row = concept_index.get(str(cid))
    if not row:
        return None
    name = row["name"]
    desc = row.get("description")
    return f"{name}. {desc}".strip() if desc else name


def concept_text_from_ref(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None
        return concept_text_from_id(value) or value
    if isinstance(value, (int, float)):
        return concept_text_from_id(str(value)) or str(value)
    if isinstance(value, dict):
        text = coerce_text(first_nonempty(value, NODE_TEXT_KEYS))
        if text:
            return text
        cid = first_nonempty(value, NODE_ID_KEYS)
        if cid is not None:
            return concept_text_from_id(str(cid)) or str(cid)
    return None


def normalize_pair_dict(row: dict[str, Any]) -> dict[str, Any] | None:
    label = None
    for key in LABEL_KEYS:
        if key in row:
            label = normalize_label(row.get(key))
            if label is not None:
                break

    source_id = first_nonempty(row, SOURCE_ID_KEYS)
    target_id = first_nonempty(row, TARGET_ID_KEYS)
    source_text = coerce_text(first_nonempty(row, SOURCE_TEXT_KEYS)) or concept_text_from_id(str(source_id))
    target_text = coerce_text(first_nonempty(row, TARGET_TEXT_KEYS)) or concept_text_from_id(str(target_id))

    if label is None:
        relation = coerce_text(first_nonempty(row, ["relation", "type", "rel"]))
        if relation:
            label = normalize_label(relation)

    if source_text and target_text and label is not None:
        return {
            "source_text": source_text,
            "target_text": target_text,
            "label": int(label),
            "source_id": str(source_id) if source_id is not None else None,
            "target_id": str(target_id) if target_id is not None else None,
            "raw": row,
        }

    return None


def normalize_pair_row(row: Any) -> list[dict[str, Any]]:
    if isinstance(row, dict):
        normalized = normalize_pair_dict(row)
        if normalized is not None:
            return [normalized]
        return adjacency_pairs_from_row(row)
    if isinstance(row, (list, tuple)) and len(row) >= 2:
        source_text = concept_text_from_ref(row[0])
        target_text = concept_text_from_ref(row[1])
        label = normalize_label(row[2]) if len(row) >= 3 else 1
        if source_text and target_text and label is not None:
            return [{
                "source_text": source_text,
                "target_text": target_text,
                "label": int(label),
                "source_id": None,
                "target_id": None,
                "raw": row,
            }]
    return []


def adjacency_pairs_from_row(row: dict[str, Any]) -> list[dict[str, Any]]:
    node_id = first_nonempty(row, NODE_ID_KEYS)
    node_text = concept_text_from_ref(first_nonempty(row, NODE_TEXT_KEYS)) or concept_text_from_id(str(node_id))
    if not node_text:
        return []

    out = []
    for key in PREREQ_LIST_KEYS:
        refs = row.get(key)
        if isinstance(refs, list):
            for ref in refs:
                prereq_text = concept_text_from_ref(ref)
                if prereq_text and prereq_text != node_text:
                    out.append({
                        "source_text": prereq_text,
                        "target_text": node_text,
                        "label": 1,
                        "source_id": None,
                        "target_id": str(node_id) if node_id is not None else None,
                        "raw": row,
                    })
    for key in CHILD_LIST_KEYS:
        refs = row.get(key)
        if isinstance(refs, list):
            for ref in refs:
                child_text = concept_text_from_ref(ref)
                if child_text and child_text != node_text:
                    out.append({
                        "source_text": node_text,
                        "target_text": child_text,
                        "label": 1,
                        "source_id": str(node_id) if node_id is not None else None,
                        "target_id": None,
                        "raw": row,
                    })
    return out


def walk_pairs(obj: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    if isinstance(obj, list):
        for row in obj:
            out.extend(normalize_pair_row(row))
            if isinstance(row, dict):
                for value in row.values():
                    if isinstance(value, list) and value and isinstance(value[0], (dict, list, tuple)):
                        out.extend(walk_pairs(value))
    elif isinstance(obj, dict):
        direct = normalize_pair_dict(obj)
        if direct is not None:
            out.append(direct)
        out.extend(adjacency_pairs_from_row(obj))
        for value in obj.values():
            if isinstance(value, (list, dict)):
                out.extend(walk_pairs(value))
    return out

parsed_pairs = walk_pairs(raw_cs)
print('Parsed MOOCCubeX pair rows:', len(parsed_pairs))
if parsed_pairs:
    print('Sample parsed row:', parsed_pairs[0])
else:
    if isinstance(raw_cs, list) and raw_cs:
        print('Sample raw row:', raw_cs[0])
    elif isinstance(raw_cs, dict):
        print('Sample raw top-level keys:', list(raw_cs.keys())[:20])
        first_key = next(iter(raw_cs.keys()), None)
        if first_key is not None:
            print('Sample raw value:', raw_cs[first_key])


Parsed MOOCCubeX pair rows: 492102
Sample parsed row: {'source_text': '操作命令', 'target_text': '重新启动', 'label': 1, 'source_id': None, 'target_id': None, 'raw': {'c1': '操作命令', 'c2': '重新启动', 'ground_truth': 1, 'text_predict': [0.015044071711599827, 0.984955906867981], 'graph_predict': [0.00342249171808362, 0.9965775012969971]}}


In [19]:
if not parsed_pairs:
    raise RuntimeError(
        "Could not parse any supervised pairs from MOOCCubeX cs.json after schema-aware parsing. Inspect the sample raw row printed in the previous cell."
    )


def directed_pair_key(row: dict[str, Any]) -> tuple[str, str]:
    src_key = str(row.get("source_id") or row["source_text"].strip())
    tgt_key = str(row.get("target_id") or row["target_text"].strip())
    return src_key, tgt_key


def undirected_group_key(row: dict[str, Any]) -> str:
    left, right = directed_pair_key(row)
    left, right = sorted((left, right))
    return f"{left}||{right}"


def normalize_training_row(row: dict[str, Any]) -> dict[str, Any] | None:
    src = row["source_text"].strip()
    tgt = row["target_text"].strip()
    if not src or not tgt or src == tgt:
        return None
    return {
        "source_text": src,
        "target_text": tgt,
        "label": int(row["label"]),
        "source_id": str(row.get("source_id")) if row.get("source_id") is not None else None,
        "target_id": str(row.get("target_id")) if row.get("target_id") is not None else None,
    }


def sample_negatives(rows: list[dict[str, Any]], negative_ratio: int | None, max_rows: int | None, rng: random.Random) -> list[dict[str, Any]]:
    pos_rows = [row for row in rows if row["label"] == 1]
    neg_rows = [row for row in rows if row["label"] == 0]
    rng.shuffle(pos_rows)
    rng.shuffle(neg_rows)

    if negative_ratio is None:
        kept_neg = neg_rows
    else:
        max_neg = len(neg_rows)
        if pos_rows:
            max_neg = min(max_neg, len(pos_rows) * int(negative_ratio))
        else:
            max_neg = 0
        kept_neg = neg_rows[:max_neg]

    sampled = pos_rows + kept_neg
    if max_rows is not None and len(sampled) > max_rows:
        if len(pos_rows) >= max_rows:
            sampled = pos_rows[:max_rows]
        else:
            remaining = max(0, max_rows - len(pos_rows))
            sampled = pos_rows + kept_neg[:remaining]
    rng.shuffle(sampled)
    return sampled


def stratified_group_holdout(rows: list[dict[str, Any]], holdout_ratio: float, seed: int) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not rows or holdout_ratio <= 0:
        return rows, []
    if holdout_ratio >= 1:
        return [], rows

    y = np.array([int(row["label"]) for row in rows])
    groups = np.array([undirected_group_key(row) for row in rows])
    indices = np.arange(len(rows))

    if len(np.unique(y)) < 2 or len(np.unique(groups)) < 2:
        train_idx, holdout_idx = train_test_split(indices, test_size=holdout_ratio, random_state=seed, stratify=y if len(np.unique(y)) > 1 else None)
        return [rows[i] for i in train_idx], [rows[i] for i in holdout_idx]

    n_splits = int(round(1.0 / holdout_ratio))
    n_splits = max(2, min(10, n_splits))

    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    target_size = int(round(len(rows) * holdout_ratio))
    target_pos_rate = float(y.mean())

    best_choice = None
    for train_idx, holdout_idx in splitter.split(np.zeros(len(rows)), y, groups):
        holdout_labels = y[holdout_idx]
        if holdout_labels.sum() == 0 or holdout_labels.sum() == len(holdout_labels):
            continue
        size_gap = abs(len(holdout_idx) - target_size)
        pos_gap = abs(float(holdout_labels.mean()) - target_pos_rate)
        score = (size_gap, pos_gap)
        if best_choice is None or score < best_choice[0]:
            best_choice = (score, train_idx, holdout_idx)

    if best_choice is None:
        train_idx, holdout_idx = train_test_split(indices, test_size=holdout_ratio, random_state=seed, stratify=y)
    else:
        _, train_idx, holdout_idx = best_choice

    return [rows[i] for i in train_idx], [rows[i] for i in holdout_idx]


positive_map: dict[tuple[str, str], dict[str, Any]] = {}
negative_map: dict[tuple[str, str], dict[str, Any]] = {}
all_concepts = set()
for row in parsed_pairs:
    normalized = normalize_training_row(row)
    if normalized is None:
        continue
    key = directed_pair_key(normalized)
    all_concepts.add(normalized["source_text"])
    all_concepts.add(normalized["target_text"])
    if normalized["label"] == 1:
        positive_map[key] = normalized
        negative_map.pop(key, None)
    else:
        if key not in positive_map:
            negative_map.setdefault(key, normalized)

concept_list = sorted(all_concepts)
rng = random.Random(SEED)

if not negative_map:
    for row in list(positive_map.values()):
        rev_row = {
            "source_text": row["target_text"],
            "target_text": row["source_text"],
            "label": 0,
            "source_id": row.get("target_id"),
            "target_id": row.get("source_id"),
        }
        rev_key = directed_pair_key(rev_row)
        if rev_key not in positive_map:
            negative_map[rev_key] = rev_row
    pos_rows_seed = list(positive_map.values())
    while len(negative_map) < len(positive_map) and concept_list:
        row = rng.choice(pos_rows_seed)
        corrupted_tgt = rng.choice(concept_list)
        candidate = {
            "source_text": row["source_text"],
            "target_text": corrupted_tgt,
            "label": 0,
            "source_id": row.get("source_id"),
            "target_id": None,
        }
        candidate_key = directed_pair_key(candidate)
        if candidate["source_text"] != corrupted_tgt and candidate_key not in positive_map:
            negative_map.setdefault(candidate_key, candidate)

all_rows = list(positive_map.values()) + list(negative_map.values())
rng.shuffle(all_rows)

raw_train_rows, temp_rows = stratified_group_holdout(all_rows, VAL_RATIO + TEST_RATIO, SEED)
if temp_rows and TEST_RATIO > 0:
    holdout_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    raw_val_rows, raw_test_rows = stratified_group_holdout(temp_rows, holdout_test_ratio, SEED + 1)
else:
    raw_val_rows, raw_test_rows = temp_rows, []

train_rows = sample_negatives(raw_train_rows, TRAIN_NEGATIVE_RATIO, MAX_TRAIN_ROWS, rng)
val_rows = sample_negatives(raw_val_rows, EVAL_NEGATIVE_RATIO, MAX_EVAL_ROWS, rng)
test_rows = sample_negatives(raw_test_rows, EVAL_NEGATIVE_RATIO, MAX_EVAL_ROWS, rng)

print({
    "all_rows_deduped": len(all_rows),
    "train_rows_raw": len(raw_train_rows),
    "val_rows_raw": len(raw_val_rows),
    "test_rows_raw": len(raw_test_rows),
    "train_rows_sampled": len(train_rows),
    "val_rows_sampled": len(val_rows),
    "test_rows_sampled": len(test_rows),
    "train_negative_ratio": TRAIN_NEGATIVE_RATIO,
    "eval_negative_ratio": EVAL_NEGATIVE_RATIO,
})
print("all label counts", Counter(row["label"] for row in all_rows))
print("train raw label counts", Counter(row["label"] for row in raw_train_rows))
print("val raw label counts", Counter(row["label"] for row in raw_val_rows))
print("test raw label counts", Counter(row["label"] for row in raw_test_rows))
print("train sampled label counts", Counter(row["label"] for row in train_rows))
print("val sampled label counts", Counter(row["label"] for row in val_rows))
print("test sampled label counts", Counter(row["label"] for row in test_rows))


{'all_rows_deduped': 492102, 'train_rows_raw': 393682, 'val_rows_raw': 49210, 'test_rows_raw': 49210, 'train_rows_sampled': 5220, 'val_rows_sampled': 7373, 'test_rows_sampled': 7272, 'train_negative_ratio': 8, 'eval_negative_ratio': 100}
all label counts Counter({0: 491377, 1: 725})
train raw label counts Counter({0: 393102, 1: 580})
val raw label counts Counter({0: 49137, 1: 73})
test raw label counts Counter({0: 49138, 1: 72})
train sampled label counts Counter({0: 4640, 1: 580})
val sampled label counts Counter({0: 7300, 1: 73})
test sampled label counts Counter({0: 7200, 1: 72})


## Stage 1: DeBERTa-NLI direction scoring on the project edge bundle

In [20]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda" and PREFER_BF16 and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
elif device == "cuda":
    model_dtype = torch.float32
else:
    model_dtype = torch.float32

if device == "cuda":
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception:
        pass

print("device=", device, "model_dtype=", model_dtype)
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0), "bf16_supported=", torch.cuda.is_bf16_supported())

deb_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_ID)
deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL_ID, torch_dtype=model_dtype).to(device)
deb_model.eval()

label_to_id = {str(v).lower(): int(k) for k, v in deb_model.config.id2label.items()}
entailment_id = next((i for lab, i in label_to_id.items() if "entail" in lab), None)
contradiction_id = next((i for lab, i in label_to_id.items() if "contrad" in lab), None)
neutral_id = next((i for lab, i in label_to_id.items() if "neutral" in lab), None)
if entailment_id is None or contradiction_id is None:
    raise RuntimeError(f"Cannot find entailment/contradiction labels in {deb_model.config.id2label}")
print("id2label=", deb_model.config.id2label)


def nli_pair(edge: dict[str, Any], reverse: bool = False) -> tuple[str, str]:
    source_text, target_text = pair_text(edge, reverse=reverse)
    premise = f"Source concept: {source_text}\nTarget concept: {target_text}"
    hypothesis = "Understanding the source concept is required before understanding the target concept."
    return premise, hypothesis


def score_nli_pairs(pairs: list[tuple[str, str]], batch_size: int = NLI_BATCH_SIZE) -> list[dict[str, float]]:
    out = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        premises = [p for p, h in batch]
        hypotheses = [h for p, h in batch]
        encoded = deb_tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH_NLI,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = deb_model(**encoded).logits.float()
        probs = F.softmax(logits, dim=-1)
        for row in range(probs.shape[0]):
            out.append({
                'entailment': probs[row, entailment_id].item(),
                'contradiction': probs[row, contradiction_id].item(),
                'neutral': probs[row, neutral_id].item() if neutral_id is not None else 0.0,
            })
    return out

forward_nli = score_nli_pairs([nli_pair(edge, reverse=False) for edge in clean_edges])
reverse_nli = score_nli_pairs([nli_pair(edge, reverse=True) for edge in clean_edges])
print('Scored NLI edges:', len(forward_nli))


device= cuda model_dtype= torch.bfloat16
gpu= Tesla T4 bf16_supported= True


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

id2label= {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
Scored NLI edges: 93


## Stage 2: Fine-tune mmBERT-base on MOOCCubeX CS

In [21]:
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

class PairDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row['source_text'],
            row['target_text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item['labels'] = torch.tensor(row['label'], dtype=torch.long)
        return item


def collate_fn(batch):
    keys = batch[0].keys()
    return {key: torch.stack([row[key] for row in batch]) for key in keys}

mb_tokenizer = AutoTokenizer.from_pretrained(MMBERT_MODEL_ID)
mb_model = AutoModelForSequenceClassification.from_pretrained(
    MMBERT_MODEL_ID,
    num_labels=2,
    torch_dtype=model_dtype,
    attn_implementation='sdpa',
).to(device)

peft_enabled = False
if USE_LORA:
    try:
        from peft import LoraConfig, TaskType, get_peft_model
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules='all-linear',
        )
        mb_model = get_peft_model(mb_model, lora_config)
        peft_enabled = True
        mb_model.print_trainable_parameters()
    except Exception as exc:
        print('LoRA attach failed; fallback to classifier tuning:', repr(exc))
        for name, param in mb_model.named_parameters():
            param.requires_grad = name.startswith('classifier') or 'score' in name

train_ds = PairDataset(train_rows, mb_tokenizer, MAX_LENGTH_FT)
val_ds = PairDataset(val_rows, mb_tokenizer, MAX_LENGTH_FT)
test_ds = PairDataset(test_rows, mb_tokenizer, MAX_LENGTH_FT) if test_rows else None
train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn) if test_ds is not None else None

train_label_counts = Counter(row['label'] for row in train_rows)
if USE_CLASS_WEIGHTS:
    class_weights = torch.tensor([
        len(train_rows) / (2.0 * max(1, train_label_counts.get(0, 0))),
        len(train_rows) / (2.0 * max(1, train_label_counts.get(1, 0))),
    ], dtype=torch.float32, device=device)
else:
    class_weights = None
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

scaler = None
use_amp = (device == "cuda" and model_dtype == torch.float16)
if use_amp:
    scaler = torch.amp.GradScaler("cuda")


optim_params = [param for param in mb_model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(optim_params, lr=LR, weight_decay=WEIGHT_DECAY)
num_train_steps = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * num_train_steps)),
    num_training_steps=num_train_steps,
)

print('train batches', len(train_loader), 'val batches', len(val_loader), 'test batches', len(test_loader) if test_loader is not None else 0, 'num_train_steps', num_train_steps)
print('train label counts', dict(train_label_counts))
print('class_weights', class_weights.detach().cpu().tolist() if class_weights is not None else None)


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA attach failed; fallback to classifier tuning: ImportError('Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported')
train batches 1305 val batches 922 test batches 909 num_train_steps 981
train label counts {0: 4640, 1: 580}
class_weights [0.5625, 4.5]


In [22]:
def pick_best_threshold(labels: list[int], probs: list[float], default_threshold: float = 0.5) -> tuple[float, float]:
    labels_np = np.asarray(labels)
    probs_np = np.asarray(probs, dtype=np.float64)
    finite_mask = np.isfinite(probs_np)
    labels_np = labels_np[finite_mask]
    probs_np = probs_np[finite_mask]
    if labels_np.size == 0 or len(np.unique(labels_np)) < 2:
        return default_threshold, 0.0
    precisions, recalls, thresholds = precision_recall_curve(labels_np, probs_np)
    if thresholds.size == 0:
        return default_threshold, 0.0
    f1_scores = (2 * precisions[:-1] * recalls[:-1]) / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


def run_eval(model, loader, threshold: float = 0.5, tune_threshold: bool = False):
    model.eval()
    all_labels = []
    all_probs = []
    total_loss = 0.0
    skipped_batches = 0
    with torch.no_grad():
        eval_iter = tqdm(loader, desc='eval', leave=False) if SHOW_PROGRESS else loader
        for batch in eval_iter:
            labels = batch['labels'].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            output = model(**inputs)
            logits = output.logits.float()
            if not torch.isfinite(logits).all():
                skipped_batches += 1
                if SKIP_NONFINITE_EVAL_BATCHES:
                    continue
                raise RuntimeError('Non-finite logits detected during eval')
            loss_val = loss_fn(logits, labels)
            if not torch.isfinite(loss_val):
                skipped_batches += 1
                if SKIP_NONFINITE_EVAL_BATCHES:
                    continue
                raise RuntimeError('Non-finite eval loss detected')
            total_loss += loss_val.item()
            probs = F.softmax(logits, dim=-1)[:, 1]
            probs_np = probs.detach().cpu().numpy()
            finite_mask = np.isfinite(probs_np)
            if not np.all(finite_mask):
                skipped_batches += 1
                probs_np = probs_np[finite_mask]
                labels_np = labels.detach().cpu().numpy()[finite_mask]
            else:
                labels_np = labels.detach().cpu().numpy()
            all_probs.extend(probs_np.tolist())
            all_labels.extend(labels_np.tolist())

    if not all_labels:
        return {
            'loss': float('nan'),
            'accuracy': float('nan'),
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'ap': float('nan'),
            'threshold': float(threshold),
            'positive_rate': 0.0,
            'skipped_batches': skipped_batches,
        }

    tuned_threshold = float(threshold)
    tuned_f1 = None
    if tune_threshold:
        tuned_threshold, tuned_f1 = pick_best_threshold(all_labels, all_probs, default_threshold=threshold)

    all_probs_np = np.asarray(all_probs, dtype=np.float64)
    all_preds = (all_probs_np >= tuned_threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    ap = average_precision_score(all_labels, all_probs_np) if len(set(all_labels)) > 1 else float('nan')
    metrics = {
        'loss': total_loss / max(1, len(loader) - skipped_batches),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ap': ap,
        'threshold': tuned_threshold,
        'positive_rate': float(np.mean(all_labels)) if all_labels else 0.0,
        'skipped_batches': skipped_batches,
    }
    if tuned_f1 is not None:
        metrics['threshold_f1'] = tuned_f1
    return metrics


best_state = None
best_f1 = -1.0
best_threshold = 0.5
best_val_metrics = None
history = []
global_step = 0

for epoch in range(EPOCHS):
    mb_model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    skipped_train_steps = 0
    train_iter = tqdm(train_loader, desc=f'train epoch {epoch+1}/{EPOCHS}', leave=False) if SHOW_PROGRESS else train_loader
    for step, batch in enumerate(train_iter, start=1):
        labels = batch['labels'].to(device)
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

        output = mb_model(**inputs)
        logits = output.logits.float()
        loss_raw = loss_fn(logits, labels)
        if not torch.isfinite(loss_raw):
            skipped_train_steps += 1
            optimizer.zero_grad(set_to_none=True)
            print(f'Skipping non-finite training loss at epoch={epoch+1} step={step}')
            continue

        loss = loss_raw / GRAD_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(optim_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if SHOW_PROGRESS and hasattr(train_iter, 'set_postfix'):
            train_iter.set_postfix(loss=f'{running_loss / max(1, step):.4f}', gs=global_step, skip=skipped_train_steps)

    metrics = run_eval(mb_model, val_loader, threshold=best_threshold, tune_threshold=TUNE_THRESHOLD_ON_VAL)
    metrics['epoch'] = epoch + 1
    metrics['train_loss'] = running_loss / max(1, len(train_loader) - skipped_train_steps)
    metrics['skipped_train_steps'] = skipped_train_steps
    history.append(metrics)
    print(metrics)

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_threshold = metrics['threshold']
        best_val_metrics = dict(metrics)
        if peft_enabled:
            best_state = {k: v.detach().cpu() for k, v in mb_model.state_dict().items() if 'lora_' in k or 'classifier' in k or 'score' in k}
        else:
            best_state = {k: v.detach().cpu() for k, v in mb_model.state_dict().items() if v.requires_grad or k.startswith('classifier')}

if best_state is not None:
    current = mb_model.state_dict()
    current.update(best_state)
    mb_model.load_state_dict(current, strict=False)

test_metrics = run_eval(mb_model, test_loader, threshold=best_threshold, tune_threshold=False) if test_loader is not None else None
print('best_threshold=', best_threshold)
print('best_val_metrics=', best_val_metrics)
print('test_metrics=', test_metrics)


train epoch 1/3:   0%|          | 0/1305 [00:00<?, ?it/s]

eval:   0%|          | 0/922 [00:00<?, ?it/s]

{'loss': 0.3690285910074468, 'accuracy': 0.9842669198426692, 'precision': 0.15873015873015872, 'recall': 0.136986301369863, 'f1': 0.14705882352941177, 'ap': 0.06326647168612005, 'threshold': 0.7371581792831421, 'positive_rate': 0.009900990099009901, 'skipped_batches': 0, 'threshold_f1': 0.14705882352941177, 'epoch': 1, 'train_loss': 0.700954282329457, 'skipped_train_steps': 0}


train epoch 2/3:   0%|          | 0/1305 [00:00<?, ?it/s]

eval:   0%|          | 0/922 [00:00<?, ?it/s]

...

train epoch 3/3:   0%|          | 0/1305 [00:00<?, ?it/s]

eval:   0%|          | 0/922 [00:00<?, ?it/s]

{'loss': 0.3286293551907622, 'accuracy': 0.9856232198562322, 'precision': 0.18867924528301888, 'recall': 0.136986301369863, 'f1': 0.15873015873015872, 'ap': 0.07076976981629075, 'threshold': 0.7248703241348267, 'positive_rate': 0.009900990099009901, 'skipped_batches': 0, 'threshold_f1': 0.15873015873015872, 'epoch': 3, 'train_loss': 0.5013706969061574, 'skipped_train_steps': 0}


eval:   0%|          | 0/909 [00:00<?, ?it/s]

best_threshold= 0.7248703241348267
best_val_metrics= {'loss': 0.3286293551907622, 'accuracy': 0.9856232198562322, 'precision': 0.18867924528301888, 'recall': 0.136986301369863, 'f1': 0.15873015873015872, 'ap': 0.07076976981629075, 'threshold': 0.7248703241348267, 'positive_rate': 0.009900990099009901, 'skipped_batches': 0, 'threshold_f1': 0.15873015873015872, 'epoch': 3, 'train_loss': 0.5013706969061574, 'skipped_train_steps': 0}
test_metrics= {'loss': 0.33658071119662036, 'accuracy': 0.9841859185918592, 'precision': 0.12280701754385964, 'recall': 0.09722222222222222, 'f1': 0.10852713178294573, 'ap': 0.05592496073254375, 'threshold': 0.7248703241348267, 'positive_rate': 0.009900990099009901, 'skipped_batches': 0}


## Stage 3: Score project edges with the fine-tuned mmBERT-base

In [23]:
def score_strength_rows(rows: list[dict[str, str]], batch_size: int = EVAL_BATCH_SIZE) -> list[float]:
    mb_model.eval()
    out = []
    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]
        encoded = mb_tokenizer(
            [row['source_text'] for row in batch_rows],
            [row['target_text'] for row in batch_rows],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH_FT,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            logits = mb_model(**encoded).logits.float()
        if not torch.isfinite(logits).all():
            probs = torch.full((len(batch_rows),), 0.5, dtype=torch.float32, device=logits.device)
        else:
            probs = F.softmax(logits, dim=-1)[:, 1]
            probs = torch.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
        out.extend(probs.detach().cpu().tolist())
    return out

forward_strength_rows = []
reverse_strength_rows = []
for edge in clean_edges:
    src_text, tgt_text = pair_text(edge, reverse=False)
    forward_strength_rows.append({'source_text': src_text, 'target_text': tgt_text})
    rev_src_text, rev_tgt_text = pair_text(edge, reverse=True)
    reverse_strength_rows.append({'source_text': rev_src_text, 'target_text': rev_tgt_text})

forward_strength = score_strength_rows(forward_strength_rows)
reverse_strength = score_strength_rows(reverse_strength_rows)
print('Scored strength edges:', len(forward_strength))


Scored strength edges: 93


## Stage 4: Combine NLI direction + fine-tuned strength into one output artifact

In [24]:
def direction_confidence_from_margin(margin: float) -> str:
    abs_margin = abs(margin)
    if abs_margin >= 0.30:
        return 'high'
    if abs_margin >= 0.15:
        return 'medium'
    return 'low'


def combo_decision(nli_margin: float, ft_forward: float, ft_reverse: float) -> tuple[str, str]:
    ft_margin = ft_forward - ft_reverse
    combo_margin = 0.6 * nli_margin + 0.4 * ft_margin
    best_strength = max(ft_forward, ft_reverse)

    if combo_margin >= COMBO_MARGIN_THRESHOLD and ft_forward >= FT_STRONG_THRESHOLD:
        return 'keep', direction_confidence_from_margin(combo_margin)
    if combo_margin <= -COMBO_MARGIN_THRESHOLD and ft_reverse >= FT_STRONG_THRESHOLD:
        return 'flip', direction_confidence_from_margin(combo_margin)
    if best_strength < FT_WEAK_THRESHOLD:
        return 'prune', 'low'
    return 'defer', direction_confidence_from_margin(combo_margin)

edge_scores = []
for idx, edge in enumerate(clean_edges):
    nli_f = forward_nli[idx]
    nli_r = reverse_nli[idx]
    nli_margin = nli_f['entailment'] - nli_r['entailment']
    ft_f = float(forward_strength[idx])
    ft_r = float(reverse_strength[idx])
    ft_margin = ft_f - ft_r
    decision, decision_conf = combo_decision(nli_margin, ft_f, ft_r)

    edge_scores.append({
        'source_kp_id': edge['source_kp_id'],
        'target_kp_id': edge['target_kp_id'],
        'deberta_forward_entailment': nli_f['entailment'],
        'deberta_forward_contradiction': nli_f['contradiction'],
        'deberta_reverse_entailment': nli_r['entailment'],
        'deberta_reverse_contradiction': nli_r['contradiction'],
        'deberta_direction_margin': nli_margin,
        'deberta_direction_confidence': direction_confidence_from_margin(nli_margin),
        'mmbert_forward_prereq_probability': ft_f,
        'mmbert_reverse_prereq_probability': ft_r,
        'mmbert_strength_margin': ft_margin,
        'combo_decision': decision,
        'combo_decision_confidence': decision_conf,
        'input_edge_scope': edge.get('edge_scope'),
        'input_keep_confidence': edge.get('keep_confidence'),
        'input_review_status': edge.get('review_status'),
    })

summary = Counter(row['combo_decision'] for row in edge_scores)
print('decision_summary', dict(summary))
print('sample', json.dumps(edge_scores[0], ensure_ascii=False, indent=2)[:1200] if edge_scores else 'NONE')


decision_summary {'defer': 81, 'prune': 12}
sample {
  "source_kp_id": "kp_3d_datasets_and_ai_geometry_task_space",
  "target_kp_id": "kp_multi_view_and_voxel_based_3d_deep_learning",
  "deberta_forward_entailment": 0.0038234079256653786,
  "deberta_forward_contradiction": 0.00028132833540439606,
  "deberta_reverse_entailment": 0.0027999759186059237,
  "deberta_reverse_contradiction": 0.00033967557828873396,
  "deberta_direction_margin": 0.001023432007059455,
  "deberta_direction_confidence": "low",
  "mmbert_forward_prereq_probability": 0.41679665446281433,
  "mmbert_reverse_prereq_probability": 0.438785195350647,
  "mmbert_strength_margin": -0.02198854088783264,
  "combo_decision": "defer",
  "combo_decision_confidence": "low",
  "input_edge_scope": "intra_course",
  "input_keep_confidence": "medium",
  "input_review_status": "optional"
}


In [25]:
output_payload = {
    'input_source': {
        'drive_url': DRIVE_URL,
        'input_path': INPUT_PATH,
        'clean_candidate_edges': len(clean_edges),
        'pruned_edges': len(pruned_edges),
        'global_kps': len(global_kps),
    },
    'training_source': {
        'dataset': 'MOOCCubeX prerequisite CS',
        'cs_url': MOOCCUBEX_CS_URL,
        'concept_url': MOOCCUBEX_CONCEPT_URL,
        'all_rows_deduped': len(all_rows),
        'train_rows_raw': len(raw_train_rows),
        'val_rows_raw': len(raw_val_rows),
        'test_rows_raw': len(raw_test_rows),
        'train_rows_sampled': len(train_rows),
        'val_rows_sampled': len(val_rows),
        'test_rows_sampled': len(test_rows),
        'label_distribution_all': dict(Counter(row['label'] for row in all_rows)),
        'label_distribution_train_raw': dict(Counter(row['label'] for row in raw_train_rows)),
        'label_distribution_val_raw': dict(Counter(row['label'] for row in raw_val_rows)),
        'label_distribution_test_raw': dict(Counter(row['label'] for row in raw_test_rows)),
        'label_distribution_train_sampled': dict(Counter(row['label'] for row in train_rows)),
        'label_distribution_val_sampled': dict(Counter(row['label'] for row in val_rows)),
        'label_distribution_test_sampled': dict(Counter(row['label'] for row in test_rows)),
        'train_negative_ratio': TRAIN_NEGATIVE_RATIO,
        'eval_negative_ratio': EVAL_NEGATIVE_RATIO,
        'max_train_rows': MAX_TRAIN_ROWS,
        'max_eval_rows': MAX_EVAL_ROWS,
        'use_class_weights': USE_CLASS_WEIGHTS,
        'best_threshold': best_threshold,
        'best_val_metrics': best_val_metrics,
        'test_metrics': test_metrics,
    },
    'models': {
        'deberta_nli_model_id': DEBERTA_MODEL_ID,
        'mmbert_model_id': MMBERT_MODEL_ID,
        'mmbert_peft_enabled': peft_enabled,
    },
    'mmbert_training_history': history,
    'edge_scores': edge_scores,
}

Path(OUTPUT_PATH).write_text(json.dumps(output_payload, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
print('Wrote', OUTPUT_PATH)
print('Output MB', round(Path(OUTPUT_PATH).stat().st_size / (1024 * 1024), 2))


Wrote /content/deberta_mmbert_mooccubex_cs_scores.json
Output MB 0.08


In [28]:
# Optional save to Drive + optional browser download
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copyfile(OUTPUT_PATH, DRIVE_OUTPUT_PATH)
        print('Saved to Drive:', DRIVE_OUTPUT_PATH)
    except Exception as exc:
        print('Drive save skipped:', repr(exc))

try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except Exception as exc:
    print('Download UI unavailable in this environment:', repr(exc))


Mounted at /content/drive
Saved to Drive: /content/drive/MyDrive/deberta_mmbert_mooccubex_cs_scores.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>